# Test Predict
Uses last MLFlow experiment of the inference model.

In [1]:
import os
import sys
import pickle
from pathlib import Path

import mlflow
from mlflow.tracking import MlflowClient
from scipy.sparse import hstack

f:\HSE\year-project\toxic-messages-data-platform\toxic-messages-handling-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DEFAULT_MODEL_CONFIG = PROJECT_ROOT / 'src' / 'config_s3.json'
if 'MODEL_CONFIG' not in os.environ and DEFAULT_MODEL_CONFIG.exists():
    os.environ['MODEL_CONFIG'] = str(DEFAULT_MODEL_CONFIG)

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from services.utils import load_config
from services.text_preprocessor import LinearSVMPreprocessorSI

In [3]:
TRACKING_URI = 'http://localhost:5000'
EXPERIMENT_NAME = 'linear-svc-binary-inference-eq'
PRD_TAG_KEY = 'tag'
PRD_TAG_VALUE = 'PRD'

MODEL_ARTIFACT_PATH = 'model_weights/linear_svc_model.pkl'
ENCODER_ARTIFACT_PATH = 'encoder/linear_svc_vectorizer.pkl'
CONFIG_PATH = os.environ.get('MODEL_CONFIG', 'src/config_s3.json')

In [4]:
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient(tracking_uri=TRACKING_URI)

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f'Experiment not found: {EXPERIMENT_NAME}')

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.{PRD_TAG_KEY} = '{PRD_TAG_VALUE}' and attributes.status = 'FINISHED'",
    order_by=['attributes.start_time DESC'],
    max_results=1,
)

if not runs:
    raise ValueError('No FINISHED runs found with tag PRD')

prd_run = runs[0]
run_id = prd_run.info.run_id
print('Using PRD run_id:', run_id)

Using PRD run_id: 63cc099790374a6ea327f93a45c953d0


In [5]:
local_model_path = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path=MODEL_ARTIFACT_PATH,
)
local_encoder_path = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path=ENCODER_ARTIFACT_PATH,
)

with open(local_model_path, 'rb') as f:
    model = pickle.load(f)

with open(local_encoder_path, 'rb') as f:
    encoder = pickle.load(f)

full_config = load_config(CONFIG_PATH)
predictor_config = full_config['predictors'][0]
preprocessor = LinearSVMPreprocessorSI(config=predictor_config)

print('Model and encoder loaded successfully')
print('model artifact:', local_model_path)
print('encoder artifact:', local_encoder_path)

enc_emoj_file models\v0\encoding_emoji.json
Model and encoder loaded successfully
model artifact: C:\Temp\tmpb23fd24z\linear_svc_model.pkl
encoder artifact: C:\Temp\tmpok6re2mj\linear_svc_vectorizer.pkl


In [6]:
def predict_one(text: str):
    text_preprocessed, num_features = preprocessor.preprocess(str(text))
    x_text = encoder.transform([text_preprocessed])
    x_input = hstack([x_text, num_features]) if num_features is not None else x_text

    pred_int = int(model.predict(x_input)[0])
    pred_label = 'toxic' if pred_int == 1 else 'non_toxic'

    score = None
    if hasattr(model, 'decision_function'):
        score = float(model.decision_function(x_input)[0])

    print('text:', text)
    print('prediction_int:', pred_int)
    print('prediction_label:', pred_label)
    print('decision_score:', score)

    return pred_int, pred_label, score

In [7]:
test_text = 'Хватит уже долбать ллм, иди бурить скважины'
pred_int, pred_label, score = predict_one(test_text)


text: Хватит уже долбать ллм, иди бурить скважины
prediction_int: 1
prediction_label: toxic
decision_score: 0.01330515523933129


In [8]:
test_text = 'Я понимаю, что не хочется, но давай, хорошка, пора.'
pred_int, pred_label, score = predict_one(test_text)


text: Я понимаю, что не хочется, но давай, хорошка, пора.
prediction_int: 0
prediction_label: non_toxic
decision_score: -0.13611795953803707
